In [83]:
import matplotlib.pyplot as plt
import numpy as np
from anemone.physics import *
from numpy.linalg import solve
from numpy.linalg import det

In [84]:
sigma = np.array([0.1, 1., 0.1])
thick = np.array([20, 40])
mu = np.ones_like(sigma)*mu0
epsilon = np.ones_like(sigma)*epsilon0
omega = 100
wave_length=0.1


rte = refl_transverse_electric(sigma, thick, omega, wave_length, mu=mu, epsilon=epsilon)

In [85]:
rte

(-6.843951690468184e-07-0.00036592403551111553j)

In [86]:
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)
zn = calc_zy(sigma, omega, mu=mu, epsilon=epsilon)[0]

adm_intrinsic = nt.sqrt(lambda2 - k2)

u0 = freespace_u0(omega, wave_length)

In [87]:
u0

(0.09999999999944369+0j)

In [88]:
un=adm_intrinsic
print(un)

[0.10000002+6.28318407e-05j 0.10000197+6.28306129e-04j
 0.10000002+6.28318407e-05j]


In [89]:
z = np.cumsum(thick)

In [90]:
%%timeit

lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)
zn = calc_zy(sigma, omega, mu=mu, epsilon=epsilon)[0]

adm_intrinsic = nt.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)
un=adm_intrinsic


A = np.array([[0, 1, 0, 0, 0, 0, 0, 0],
[1, 1, -1, -1, 0, 0, 0, 0],
[0, 0, np.exp(un[0]*z[0]), np.exp(-un[0]*z[0]), -np.exp(un[1]*z[0]), -np.exp(-un[1]*z[0]), 0, 0],
[0, 0, 0, 0, np.exp(un[1]*z[1]), np.exp(-un[1]*z[1]), -np.exp(un[2]*z[1]), -np.exp(-un[2]*z[1])],
[u0, -u0, -un[0], un[0], 0, 0, 0, 0],
[0, 0, un[0]*np.exp(un[0]*z[0]), -un[0]*np.exp(-un[0]*z[0]), -un[1]*np.exp(un[1]*z[0]), un[1]*np.exp(-un[1]*z[0]), 0, 0],
[0, 0, 0, 0, un[1]*np.exp(un[1]*z[1]), -un[1]*np.exp(-un[1]*z[1]), -un[2]*np.exp(un[2]*z[1]), un[2]*np.exp(-un[2]*z[1])],
[0, 0, 0, 0, 0, 0, 1, 0]])
ans = np.array([1., 0, 0, 0, 0, 0, 0, 0])
ab = solve(A, ans)

208 µs ± 15.8 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [91]:
%%timeit
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)
zn = calc_zy(sigma, omega, mu=mu, epsilon=epsilon)[0]

adm_intrinsic = nt.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)
un=adm_intrinsic

exp_unh_p = np.exp(un[:-1]*thick)
exp_unh_m = 1/exp_unh_p


A = np.array([[0, 1, 0, 0, 0, 0, 0, 0],
[1, 1, -1, -1, 0, 0, 0, 0],
[0, 0, exp_unh_p[0], exp_unh_m[0], -1, -1, 0, 0],
[0, 0, 0, 0, exp_unh_p[1], exp_unh_m[1], -1, -1],
[u0, -u0, -un[0], un[0], 0, 0, 0, 0],
[0, 0, un[0]*exp_unh_p[0], -un[0]*exp_unh_m[0], -un[1], un[1], 0, 0],
[0, 0, 0, 0, un[1]*exp_unh_p[1], -un[1]*exp_unh_m[1], -un[2], un[2]],
[0, 0, 0, 0, 0, 0, 1, 0]])
ans = np.array([1., 0, 0, 0, 0, 0, 0, 0])

A1 = A*1.
A1[0] = ans

rte1 = det(A1)/det(A)

64.5 µs ± 2.7 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [92]:
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p


A = np.array([
    [ 0,   1,              0,               0,              0,               0,      0,     0],
    [ 1,   1,             -1,              -1,              0,               0,      0,     0],
    [u0, -u0,         -un[0],           un[0],              0,               0,      0,     0],
    [ 0,   0,       euh_p[0],        euh_m[0],             -1,              -1,      0,     0],
    [ 0,   0, un[0]*euh_p[0], -un[0]*euh_m[0],         -un[1],           un[1],      0,     0],
    [ 0,   0,              0,               0,       euh_p[1],        euh_m[1],     -1,    -1],
    [ 0,   0,              0,               0, un[1]*euh_p[1], -un[1]*euh_m[1], -un[2], un[2]],
    [ 0,   0,              0,               0,              0,               0,      1,     0]])

A = np.array([
    [ 0,   1,              0,                 0,              0,                 0,            0,           0],
    [ 1,   1,             -1,                -1,              0,                 0,            0,           0],
    [ 0, -2*u0,    -un[0]+u0,          un[0]+u0,              0,                 0,            0,           0],
    [ 0,   0,       euh_p[0],          euh_m[0],             -1,                -1,            0,           0],
    [ 0,   0,              0, -2*un[0]*euh_m[0],   -un[1]+un[0],       un[1]+un[0],            0,           0],
    [ 0,   0,              0,                 0,       euh_p[1],          euh_m[1],           -1,          -1],
    [ 0,   0,              0,                 0,              0, -2*un[1]*euh_m[1], -un[2]+un[1], un[2]+un[1]],
    [ 0,   0,              0,                 0,              0,                 0,            1,           0]
    ])


ans = np.array([1., 0, 0, 0, 0, 0, 0, 0])

A1 = A*1.
A1[0] = ans

rte1 = det(A1)/det(A)
print(rte)
print(rte1)

(-6.843951690468184e-07-0.00036592403551111553j)
(-6.843951692217494e-07-0.00036592403551111505j)


In [93]:
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p


A = np.array([
    [ 0,   1,              0,                 0,              0,                 0,            0],
    [ 1,   1,             -1,                -1,              0,                 0,            0],
    [ 0, -2*u0,    -un[0]+u0,          un[0]+u0,              0,                 0,            0],
    [ 0,   0,       euh_p[0],          euh_m[0],             -1,                -1,            0],
    [ 0,   0,              0, -2*un[0]*euh_m[0],   -un[1]+un[0],       un[1]+un[0],            0],
    [ 0,   0,              0,                 0,       euh_p[1],          euh_m[1],           -1],
    [ 0,   0,              0,                 0,              0, -2*un[1]*euh_m[1],  un[2]+un[1]]
            ])


ans = np.array([1., 0, 0, 0, 0, 0, 0])

A1 = A*1.
A1[0] = ans

rte1 = det(A1)/det(A)
print(rte)
print(rte1)

(-6.843951690468184e-07-0.00036592403551111553j)
(-6.843951692217494e-07-0.00036592403551111505j)


In [94]:
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p


A = np.array([
    [ 1,          -1,                -1,              0,                 0,            0],
    [ 0,    u0-un[0],          un[0]+u0,              0,                 0,            0],
    [ 0,    euh_p[0],          euh_m[0],             -1,                -1,            0],
    [ 0,           0, -2*un[0]*euh_m[0],   -un[1]+un[0],       un[1]+un[0],            0],
    [ 0,           0,                 0,       euh_p[1],          euh_m[1],           -1],
    [ 0,           0,                 0,              0, -2*un[1]*euh_m[1],  un[2]+un[1]]
            ])


ans = np.array([-1, 2*u0, 0, 0, 0, 0])

A1 = A*1.
A1[:, 0] = ans

rte1 = det(A1)/det(A)

ab = solve(A, ans)

print(rte)
print(rte1)
print(ab[0])

(-6.843951690468184e-07-0.00036592403551111553j)
(-6.843951692217494e-07-0.00036592403551111505j)
(-6.843951692998101e-07-0.0003659240355111151j)


In [149]:
# try to get things into an upper triangle matrix

lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p


A0 = np.array([
    [ 1,          -1,                                      -1,              0,                 0,            0],
    [ 0,    u0-un[0],                                un[0]+u0,              0,                 0,            0],
    [ 0,           0, euh_m[0]-euh_p[0]*(un[0]+u0)/(u0-un[0]),             -1,                -1,            0],
    [ 0,           0,                       -2*un[0]*euh_m[0],   -un[1]+un[0],       un[1]+un[0],            0],
    [ 0,           0,                                       0,       euh_p[1],          euh_m[1],           -1],
    [ 0,           0,                                       0,              0, -2*un[1]*euh_m[1],  un[2]+un[1]]
            ])


ans = np.array([-1, 2*u0, -euh_p[0]*2*u0/(u0-un[0]), 0, 0, 0])

A1 = A*1.
A1[:, 0] = ans

diag1 = np.stack((euh_p, -2*un[:-1]*euh_m)).T.flatten()


rte1 = det(A1)/det(A)

ab = solve(A, ans)

print(rte)
print(rte1)
print(ab[0])

(-6.843951690468184e-07-0.00036592403551111553j)
(-6.843951707460398e-07-0.00036592403592597395j)
(-6.843951697532512e-07-0.000365924035486901j)


In [151]:
A0=A*1.

In [196]:
%%timeit

# try to get things into an upper triangle matrix
lambda2 = wave_length**2
k2 = calc_wavenumber(sigma, omega, mu=mu, epsilon=epsilon, sqrt=False)

un = np.sqrt(lambda2 - k2)
u0 = freespace_u0(omega, wave_length)

# un0 = np.insert(un,0,u0)


euh_p = np.exp(un[:-1]*thick)
euh_m = 1/euh_p

d_un = un[1:]-un[:-1]
m_un = un[1:]+un[:-1]

diag1 = np.stack((euh_m, 2*un[:-1]*euh_p)).T.flatten()
diag1 = np.insert(diag1, 0, u0)

diag2 = np.stack((d_un*euh_p[:-1]/m_un, d_un)).T.flatten()
diag2 = np.insert(diag2, 0, np.array([1-u0/un[0], un[0]]))

diag3 = np.stack((-2*un[1:]/m_un, -m_un)).T.flatten()
diag3 = np.insert(diag3, 0, np.array([-2, -un[0]]))[:-1]

A = np.diag(diag1[:-1], k=1) + np.diag(diag2[:-1])+np.diag(diag3[:-1],k=-1)

ans = np.array([-1+u0/un[0], u0, 0, 0, 0])

A1 = A*1.
A1[:, 0] = ans


rte1 = det(A1)/det(A)

162 µs ± 12.8 µs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [190]:
print(det(A))
print(det(A1))

(-4.293230610736228e-06-0.0061901415153178315j)
(0.00010340307707928015+0.006497376736223743j)


In [ ]:

ans = np.array([-1, 2*u0, -euh_p[0]*2*u0/(u0-un[0]), 0, 0, 0])

A1 = A*1.
A1[:, 0] = ans



rte1 = det(A1)/det(A)
ab = solve(A, ans)

print(rte)
print(rte1)
print(ab[0])

In [97]:
detA = un[-1]*np.prod(un[:-1]*exp_unh_m)
print(detA)

(2.4782149823830217e-06-4.671837625041902e-08j)


In [98]:
print(det(A))

(3.2259663668876426+0.10847954095765923j)


In [99]:
print(det(A))

(3.2259663668876426+0.10847954095765923j)


In [100]:
rte = refl_transverse_electric(sigma, thick, omega, wave_length)#, mu=mu, epsilon=epsilon)

In [101]:
from numpy.linalg import solve

In [102]:
ab = solve(A, ans)

In [103]:
print(ab)

[-1.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j  0.+0.j]


In [104]:
(ab[0]/ab[1])**2

C:\Users\au646858\AppData\Local\Temp\ipykernel_516\2338927527.py:1: RuntimeWarning: divide by zero encountered in scalar divide
  (ab[0]/ab[1])**2
C:\Users\au646858\AppData\Local\Temp\ipykernel_516\2338927527.py:1: RuntimeWarning: invalid value encountered in scalar divide
  (ab[0]/ab[1])**2


(nan+nanj)

In [105]:
print((ab[0]/ab[1])*np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0]))
print((ab[0]/ab[1])/(np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0])))

(nan+nanj)
(nan+nanj)


C:\Users\au646858\AppData\Local\Temp\ipykernel_516\441652881.py:1: RuntimeWarning: divide by zero encountered in scalar divide
  print((ab[0]/ab[1])*np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0]))
C:\Users\au646858\AppData\Local\Temp\ipykernel_516\441652881.py:1: RuntimeWarning: invalid value encountered in scalar divide
  print((ab[0]/ab[1])*np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0]))
C:\Users\au646858\AppData\Local\Temp\ipykernel_516\441652881.py:2: RuntimeWarning: divide by zero encountered in scalar divide
  print((ab[0]/ab[1])/(np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0])))
C:\Users\au646858\AppData\Local\Temp\ipykernel_516\441652881.py:2: RuntimeWarning: invalid value encountered in scalar divide
  print((ab[0]/ab[1])/(np.exp(-un[0]*z[0])*np.exp(-un[0]*z[0])))


In [106]:
print(rte1)
print(rte)

(-0.4500003512543463-0.0001986699816882026j)
(-6.843951690468184e-07-0.00036592403551111553j)


In [124]:
a = np.arange(5)+1
b = np.arange(4)+1
c = np.arange(4)+1

In [134]:
from scipy.linalg import eigvalsh_tridiagonal as eigtri

np.product(eigtri(a, b))

-48.99999999999954

In [128]:
A = np.diag(a)+np.diag(b, k=1) + np.diag(c, k=-1)

In [130]:
det(A)

-48.99999999999997

In [ ]:
def tridet(a,b,c):
    n  = len(a)-1
    if len(c)>1:
        fn = tridet(a[1:], b[1:], c[1:])
    else:
        fn